# Overfitting e hiperparametros


Usa esta plantilla cuando pidan mejorar un modelo, comparar train/test, usar cross-validation o ajustar hiperparámetros.

La idea central: si el modelo va muy bien en train y mal fuera, está memorizando.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

TARGET = "target"  # CAMBIAR

df = pd.read_csv("data.csv")
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None,
)

numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
    ]
)


In [ ]:
base_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

base_model.fit(X_train, y_train)

print("TRAIN")
print(classification_report(y_train, base_model.predict(X_train), zero_division=0))

print("TEST")
print(classification_report(y_test, base_model.predict(X_test), zero_division=0))


In [ ]:
# Ajuste sencillo de hiperparámetros.
# Mantén la grid pequeña: en examen interesa demostrar criterio, no probar 200 combinaciones.
model_for_search = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 8, 16],
    "classifier__min_samples_leaf": [1, 3, 5],
}

search = GridSearchCV(
    model_for_search,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
)

search.fit(X_train, y_train)

print("Mejores parámetros:", search.best_params_)
print("Mejor score CV:", search.best_score_)

best_model = search.best_estimator_
print(classification_report(y_test, best_model.predict(X_test), zero_division=0))


## Cómo diagnosticar

- Train alto y test bajo: overfitting.
- Train bajo y test bajo: underfitting o datos/features pobres.
- CV alto y test bajo: quizá el test tiene distribución distinta o hubo leakage.
- Una clase va mal: cambia `scoring` a algo como `f1_macro`.
